# Setup Environment

This file is used to validate the results are replicated for a small dataset across sites.
This ensures that the environment at each site is functioning the same.

In [1]:
environment = 'embed-open-data' #select from: 'duke', 'embed', 'embed-open-data', 'icad'

import warnings
from torch.serialization import SourceChangeWarning
warnings.filterwarnings("ignore", category=SourceChangeWarning)

In [ ]:
!nvidia-smi

In [3]:
import torch
torch.cuda.set_device(os.environ['TORCH_DEVICE'])

In [4]:
# imports
import pandas as pd
from tqdm import tqdm
import torch
from pathlib import Path

# only set to true the first time this was run
resample_images = False

external_validation_path = Path(os.environ['EXTERNAL_VAL_ROOT'])

sample_data_path = external_validation_path / 'sample-data'
model_path = external_validation_path / 'models'
logit_path = external_validation_path / 'logits'

assert external_validation_path.exists()
assert model_path.exists()
assert logit_path.exists()

%matplotlib inline

In [5]:
vanilla_vgg_path = model_path / '0.9504818324659497_at_epoch_60_corrected_format'
protopnet_path = model_path / '50_5push0.9209.pth'
iaiabl_path = model_path / '50_4push0.9546.pth'
bb_malig_path = model_path / '0.8701298701298701_at_epoch_16_corrected_format'

assert vanilla_vgg_path.exists()
assert protopnet_path.exists()
assert iaiabl_path.exists()
assert bb_malig_path.exists()

# Replication
Sanity check the model behavior in this environment matches the source data

### Copy a Sample of Files

In [6]:
import os
import random

source_dir = Path(os.environ['SAMPLE_DATA_PATH'])

repl_data_path = external_validation_path / 'sample-data'

# List of subfolders
subfolders = ["Circumscribed", "Indistinct", "Spiculated"]

def sample_files_and_create_symlinks(subfolder):
    orig_subfolder_path = source_dir / subfolder
    new_subfolder_path = sample_data_path / subfolder

    # Ensure the subfolder exists in the new directory
    new_subfolder_path.mkdir(parents=True, exist_ok=True)

    # Get list of files in the original subfolder
    all_files = [f for f in os.listdir(str(orig_subfolder_path)) if os.path.isfile(os.path.join(str(orig_subfolder_path), f))]

    # Sample 5 files
    sampled_files = random.sample(all_files, 5)

    # Create symbolic links for the sampled files in the new directory
    for file in sampled_files:
        orig_file_path = orig_subfolder_path / file
        new_file_link_path = new_subfolder_path / file
        os.symlink(orig_file_path, new_file_link_path)

if resample_images:
    !rm -rf $repl_data_path/*

    for subfolder in subfolders:
        sample_files_and_create_symlinks(subfolder)

# ! ls -ltrh $repl_data_path/*

# Model Predictions

In [ ]:
torch.cuda.current_device()

In [ ]:
from dataHelper import DatasetFolder
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from vanilla_vgg import Vanilla_VGG
import numpy as np

f_map_location = lambda storage, loc: storage.cuda(torch.cuda.current_device())

bb_model_malig = torch.load(bb_malig_path, map_location=f_map_location)
bb_model_malig.eval()

test_dataset = DatasetFolder(
    sample_data_path,
    loader=np.load,
    extensions=("npy",),
    transform = transforms.Compose([
        torch.from_numpy,
    ])
    )

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=20, shuffle=False,
    num_workers=4, pin_memory=False)

np.random.seed(0)
torch.random.manual_seed(0)

all_predictions = None
for model_name, model_path in [('protopnet', protopnet_path), ('iaiabl', iaiabl_path), ('vgg16', vanilla_vgg_path)]:

    print(model_name, "-------------", end="\n\n")
    model = torch.load(model_path, map_location=f_map_location)
    model.eval()

    exam_ids = []
    circ = []
    indst = []
    spic = []
    context_sizes = []
    malig_neg = []
    malig_pos = []

    for i, (image, label, patient_id) in tqdm(enumerate(test_loader)):
        if 'vgg16' == model_name:
            # image = image[:, :-1]
            pred = model.forward(image.cuda())
            malig_pred = bb_model_malig(image.cuda())
        else:
            pred = model(image.cuda())[0]
            
        for p in range(pred.shape[0]):
            circ.append(pred.detach().cpu()[p, 0].item())
            indst.append(pred.detach().cpu()[p, 1].item())
            spic.append(pred.detach().cpu()[p, 2].item())
            exam_ids.append(patient_id[p])
            context_sizes.append(100)
            
            if 'vgg16' == model_name:
                malig_neg.append(pred.detach().cpu()[p, 0].item())
                malig_pos.append(pred.detach().cpu()[p, 1].item())

    if 'vgg16' == model_name:
        preds_df = pd.DataFrame({
            'model_name': model_name,
            'image_id': exam_ids,
            'circ_prob': np.exp(circ),
            'indst_prob': np.exp(circ),
            'spic_prob': np.exp(circ),
            'context_size': context_sizes,
            'mal_pos_prob': np.exp(malig_neg),
            'mal_neg_prob': np.exp(malig_pos)
        })
    else:
        preds_df = pd.DataFrame({
            'model_name': model_name,
            'image_id': exam_ids,
            'circ_logit': circ,
            'indst_logit': indst,
            'spic_logit': spic,
            'context_size': context_sizes
        })
    
    if all_predictions is None:
        all_predictions = preds_df
    else:
        all_predictions = pd.concat([all_predictions, preds_df], sort=True)

    all_predictions.to_csv(external_validation_path/'sample-predictions.csv')

all_predictions.groupby('model_name').count()

In [ ]:
all_predictions

In [ ]:
if environment == 'duke':
    all_predictions.to_csv(external_validation_path/ 'logits' / 'duke_logits.csv', index=False)
else:
    print("not updating logits on", environment)

# Compare Logits

In [ ]:
original_predictions = pd.read_csv(external_validation_path/ 'logits' / 'duke_logits.csv')
original_predictions.groupby('model_name').count()

In [ ]:
logits = original_predictions.merge(all_predictions, on=['image_id', 'model_name'], suffixes=('_duke', '_site'))
logits.groupby('model_name').count()

In [ ]:
THRESHOLD=1e-5

match_list = []
for margin in ['circ', 'indst', 'spic']:
    for model in ['iaiabl', 'protopnet']:
        
        model_df = logits[logits['model_name'] == model]

        diff = (model_df[f'{margin}_logit_duke'] - model_df[f'{margin}_logit_site']).abs()
        matches = diff < THRESHOLD
        match_counts = matches.value_counts()

        assert len(match_counts) == 1 and match_counts[True] == 15, (model, margin, diff)

        match_list.append((model, margin, match_counts[1]))

    for model in ['vgg16']:
        model_df = logits[logits['model_name'] == model]

        diff = (model_df[f'{margin}_prob_duke'] - model_df[f'{margin}_prob_site']).abs()
        matches = diff < THRESHOLD
        match_counts = matches.value_counts()

        assert len(match_counts) == 1 and match_counts[True] == 15, (model, margin, diff)

        match_list.append((model, margin, match_counts[True]))

match_list